# Advanced Web Scraping

`01_Scraping_Fundamentals` taught the basics. This notebook levels up to the skills you need on **real projects**:

- Working with messy, unpredictable HTML
- Combining data scraped from **multiple pages** into one dataset
- **Downloading media** safely (clean filenames, no crashes)
- Handling network errors gracefully
- Saving your results for later analysis

> **Reminder:** scrape ethically — respect `robots.txt`, site terms, and rate limits. This notebook uses public practice/demo sites.

**Prerequisite:** complete **`01_Scraping_Fundamentals.ipynb`** first.

## Why Advanced Scraping is Different

The practice sites are tidy. Real sites are not:

- Elements are sometimes missing → `.get()` returns `None` and crashes your code
- Attributes change or are empty → always filter and sanitize
- Images have spaces, slashes, or no names in their filenames → clean before saving
- Servers go down or block you → every request can throw an error

The solution is the same in every case: **guard rails** — defaults, filters, sanitizing, and `try/except`.

## Setup

Imports — the same stack as before, plus `tqdm` for progress bars:

In [ ]:
import requests as req          # fetch pages
from bs4 import BeautifulSoup as bs   # parse HTML
import pandas as pd          # structure the data
import os                    # file/folder helpers

## 1. Fetching a Live Page (Optional)

Fetch the card-shop homepage and save a local copy. If you're offline, the saved copy already exists — we'll still use it in the next cells.

In [ ]:
baseUrl = "https://www.deckshop.pro/"
try:
    response = req.get(baseUrl, timeout=10)
    response.raise_for_status()
    soup = bs(response.content, "html.parser")
    with open("deskShopOffline.html", "w", encoding="utf-8") as file:
        file.write(soup.prettify())
    print("Saved deskShopOffline.html")
except Exception as e:
    print("No internet? Using the existing deskShopOffline.html.")
    print(e)

## 2. Parsing with Guard Rails

BeautifulSoup's `.get()` lets us provide a **default** when an attribute is missing. And we can **filter out** elements that don't have what we need.

Let's re-parse the countries page from the fundamentals notebook — but this time skip anything that looks like a placeholder or is empty:

In [ ]:
# Parse the offline countries page
with open("ScrapeThisSiteOffline.html", "r", encoding="utf-8") as file:
    soup = bs(file.read(), "html.parser")

# Find all country cards and keep only the ones with a real name
countryNames = [h3.text.strip() for h3 in soup.find_all("h3", class_="country-name") if h3.text.strip()]

print(f"Found {len(countryNames)} countries. First 10:")
for name in countryNames[:10]:
    print(" -", name)

## 3. Combining Multiple Pages

Real data is spread across many pages. The pattern:

1. Loop over the page URLs
2. Parse each page (here: `pd.read_html` grabs the `<table>` directly)
3. **Tag** each chunk with its page number
4. `pd.concat()` everything into one big DataFrame

The `htmlFiles/` folder already contains the 24 pages of hockey-team data from the fundamentals notebook:

In [ ]:
frames = []
for page in range(1, 4):  # combine the first 3 pages (extend to 24 for the full set)
    page_df = pd.read_html(f"htmlFiles/scrapethissitePageNo{page}.html")[0]
    page_df["Page"] = page          # remember where each row came from
    frames.append(page_df)

all_teams = pd.concat(frames, ignore_index=True)
print(f"Combined {len(frames)} pages -> {all_teams.shape[0]} rows")
all_teams.head()

## 4. Downloading Media Safely

Downloading files adds a new problem: **filenames**. Real pages contain names with spaces, slashes, or empty `alt` attributes — all of which break file saving. The safe recipe:

1. Keep only images that have both a `src` and an `alt`
2. **Sanitize** the name (`/` and spaces → `_`)
3. Prefix relative URLs with the site's base URL
4. Cap the count and use a progress bar

In [ ]:
# Parse the offline card-shop page
with open("deskShopOffline.html", "r", encoding="utf-8") as file:
    soup = bs(file.read(), "html.parser")

# Guard rails: only images that actually have a src AND an alt
targets = [img for img in soup.find_all("img") if img.get("src") and img.get("alt")]
print(f"{len(targets)} usable images on the page")

os.makedirs("ScrappedImages", exist_ok=True)
baseUrl = "https://www.deckshop.pro"

from tqdm import tqdm
try:
    for img in tqdm(targets[:10], desc="Downloading images"):  # demo: 10 images
        name = img["alt"].replace("/", "_").replace(" ", "_")   # sanitize the filename
        url = img["src"]
        if not url.startswith("http"):                           # fix relative URLs
            url = baseUrl + url
        content = req.get(url, timeout=15).content
        with open(f"ScrappedImages/{name}.png", "wb") as f:
            f.write(content)
    print("Downloaded 10 sample images to ScrappedImages/")
except Exception as e:
    print("No internet? Skipping downloads (existing images remain).")
    print(e)

## 5. Handling Errors

The most common scraping bugs are network errors. Wrap requests in `try/except` so one bad URL doesn't kill the whole script:

In [ ]:
# This domain does not exist — the request will fail.
# Notice how the program KEEPS RUNNING after we catch the error.
try:
    response = req.get("https://this-domain-does-not-exist-12345.com", timeout=5)
    print("Status:", response.status_code)
except Exception as e:
    print("Caught an error:", type(e).__name__)
    print("->", e)

print("Execution continued past the error.")

## 🎯 Key Takeaways

- Real-world HTML is messy — always use `.get(key, default)` instead of `["key"]`.
- Filter empty / placeholder elements before processing them.
- Multi-page scraping = loop + parse + tag + `pd.concat()`.
- `pd.read_html()` turns `<table>` tags straight into DataFrames.
- Sanitize filenames before saving downloads (`/` and spaces → `_`).
- Fix relative URLs by prepending the site's base URL.
- Wrap every network call in `try/except` — and let the script continue.
- Save pages to disk once and re-parse offline to avoid hammering servers.

## 🏋️ Practice Exercises

1. Extend the multi-page cell to all 24 pages (`range(1, 25)`) and report the total rows.
2. Find the team with the most wins across all combined pages.
3. Add a `Country Data` exercise: export the countries list to a CSV with pandas.
4. In the download cell, remove the `[:10]` cap and safely download every image — how many succeed?
5. Write a tiny scraper that downloads a URL list from a text file, one per line, and saves each page as `page_0.html`, `page_1.html`, ...

## 🚀 Next Steps

- **`06_Web_Automation/01_Selenium_Automation.ipynb`** — scraping pages that load content with JavaScript (requires a real browser).
- **`03_Pandas_Data_Analysis`** — analyze everything you scrape.
- **`04_File_Handling_and_IO.ipynb`** — more about saving and reading files with Python.